In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available. Select T4 GPU from Runtime settings."
    )

print("GPU:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()
print(f"Free memory: {free / 1024**3:.2f} GiB")
print(f"Total memory: {total / 1024**3:.2f} GiB")


GPU: Tesla T4
Free memory: 14.46 GiB
Total memory: 14.56 GiB


In [2]:
%pip install -q -U transformers accelerate bitsandbytes

In [3]:

import torch
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)

model.eval()

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!
GPU: Tesla T4


In [4]:


prompt = "Explain what an Embedded system is in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        use_cache=True,
    )

answer = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(answer)

del inputs, output
gc.collect()
torch.cuda.empty_cache()




An Embedded system is a dedicated computer system designed to perform one or a few specific functions. It is embedded as a part of a complete system


In [5]:


import time
import threading
import statistics
import torch
import gc

from transformers.generation.streamers import BaseStreamer


class TimingStreamer(BaseStreamer):
    def __init__(self):
        self.token_times = []
        self.prompt_received = False

    def put(self, value):
        # The first call contains the input prompt.
        if not self.prompt_received:
            self.prompt_received = True
            return

        torch.cuda.synchronize()
        self.token_times.append(time.perf_counter())

    def end(self):
        pass


def profile_generation(context_length, max_new_tokens=16):
    # Create an input with the requested number of tokens.
    seed_ids = tokenizer.encode(
        "Explain how a microcontroller works. ",
        add_special_tokens=False,
    )

    repeated_ids = (
        seed_ids * (context_length // len(seed_ids) + 1)
    )[:context_length]

    input_ids = torch.tensor(
        [repeated_ids],
        dtype=torch.long,
        device="cuda",
    )

    attention_mask = torch.ones_like(input_ids)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    baseline_memory = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()

    streamer = TimingStreamer()
    errors = []

    def generate():
        try:
            with torch.inference_mode():
                model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    streamer=streamer,
                    pad_token_id=tokenizer.eos_token_id,
                )
        except Exception as exc:
            errors.append(exc)

    torch.cuda.synchronize()
    start = time.perf_counter()

    worker = threading.Thread(target=generate)
    worker.start()
    worker.join()

    torch.cuda.synchronize()
    end = time.perf_counter()

    if errors:
        raise errors[0]

    times = streamer.token_times

    if not times:
        raise RuntimeError("No generated tokens were recorded.")

    ttft = times[0] - start

    intervals = [
        times[i] - times[i - 1]
        for i in range(1, len(times))
    ]

    mean_latency = (
        statistics.mean(intervals)
        if intervals else float("nan")
    )

    peak_memory = torch.cuda.max_memory_allocated()

    return {
        "context_tokens": context_length,
        "generated_tokens": len(times),
        "TTFT_seconds": ttft,
        "mean_inter_token_seconds": mean_latency,
        "total_generation_seconds": end - start,
        "baseline_allocated_GiB": baseline_memory / 1024**3,
        "peak_allocated_GiB": peak_memory / 1024**3,
    }


print("Profiler is ready!")


Profiler is ready!


In [6]:


test_result = profile_generation(
    context_length=128,
    max_new_tokens=4,
)

print(test_result)

{'context_tokens': 128, 'generated_tokens': 4, 'TTFT_seconds': 0.35724576099983096, 'mean_inter_token_seconds': 0.15088994400002775, 'total_generation_seconds': 0.8104660589997366, 'baseline_allocated_GiB': 3.866405487060547, 'peak_allocated_GiB': 4.0145416259765625}


In [7]:
import pandas as pd
import gc
import torch

context_lengths = [1024, 2048, 4096, 8192]
results = []

for length in context_lengths:

    print(f"\nProfiling context length: {length}")

    for trial in range(1, 4):

        try:
            result = profile_generation(
                context_length=length,
                max_new_tokens=16,
            )

            results.append({
                "Context Length (Tokens)": length,
                "Trial": trial,
                "TTFT (Seconds)": result["TTFT_seconds"],
                "Inter-token Latency (Seconds)": result["mean_inter_token_seconds"],
                "VRAM Usage (GiB)": result["peak_allocated_GiB"],
            })

            print(f"Trial {trial} completed successfully.")

        except torch.cuda.OutOfMemoryError:

            print(f"Trial {trial}: OUT OF MEMORY")

            gc.collect()
            torch.cuda.empty_cache()

        except Exception as exc:

            print(f"Trial {trial}: ERROR - {exc}")

results_df = pd.DataFrame(results)

summary = (
    results_df.groupby("Context Length (Tokens)")
    .agg({
        "TTFT (Seconds)": "mean",
        "Inter-token Latency (Seconds)": "mean",
        "VRAM Usage (GiB)": "mean",
    })
    .reset_index()
)

display(summary.round(4))



Profiling context length: 1024
Trial 1 completed successfully.
Trial 2 completed successfully.
Trial 3 completed successfully.

Profiling context length: 2048
Trial 1 completed successfully.
Trial 2 completed successfully.
Trial 3 completed successfully.

Profiling context length: 4096
Trial 1 completed successfully.
Trial 2 completed successfully.
Trial 3 completed successfully.

Profiling context length: 8192
Trial 1: OUT OF MEMORY
Trial 2: OUT OF MEMORY
Trial 3: OUT OF MEMORY


,Context Length (Tokens),TTFT (Seconds),Inter-token Latency (Seconds),VRAM Usage (GiB)
0,1024,1.1119,0.1157,4.3866
1,2048,2.5800,0.1164,5.4692
2,4096,7.7580,0.1195,9.3453


In [8]:
golden_cases = [
    {
        "id": 1,
        "question": "What is an embedded system?",
        "reference": "An embedded system is a computer system designed to perform a specific task within a larger device.",
        "required_concepts": [["computer", "computing system"], ["specific task"], ["device", "larger system"]]
    },
    {
        "id": 2,
        "question": "What is a microcontroller?",
        "reference": "A microcontroller is a small computer on a single chip that contains a CPU, memory, and peripherals.",
        "required_concepts": [["single chip", "chip"], ["cpu", "processor"], ["memory"], ["peripherals", "input output"]]
    },
    {
        "id": 3,
        "question": "What is GPIO?",
        "reference": "GPIO stands for General Purpose Input Output. It allows a microcontroller to communicate with external devices.",
        "required_concepts": [["general purpose input output", "gpio"], ["input"], ["output"], ["external devices", "peripherals"]]
    },
    {
        "id": 4,
        "question": "What is the purpose of the GPIO MODER register in STM32?",
        "reference": "MODER configures each GPIO pin as input, output, alternate function, or analog.",
        "required_concepts": [["configures", "sets"], ["input"], ["output"], ["alternate function"], ["analog"]]
    },
    {
        "id": 5,
        "question": "What does the GPIO ODR register do?",
        "reference": "The Output Data Register controls the output state of GPIO pins.",
        "required_concepts": [["output data register", "odr"], ["output state", "output value"], ["gpio", "pin"]]
    },
    {
        "id": 6,
        "question": "What is the purpose of the GPIO IDR register?",
        "reference": "The Input Data Register reads the logic levels present on GPIO pins.",
        "required_concepts": [["input data register", "idr"], ["read", "reads"], ["logic level", "input state"], ["pin", "gpio"]]
    },
    {
        "id": 7,
        "question": "What is the purpose of the GPIO BSRR register?",
        "reference": "BSRR sets or resets individual GPIO output bits atomically.",
        "required_concepts": [["bsrr"], ["set"], ["reset"], ["atomic", "independent"]]
    },
    {
        "id": 8,
        "question": "How do you set a GPIO pin using bit manipulation?",
        "reference": "Set a bit by using bitwise OR with a mask, such as register |= (1U << pin).",
        "required_concepts": [["bitwise or", "|="], ["shift", "<<"], ["mask", "bit"]]
    },
    {
        "id": 9,
        "question": "How do you clear a specific bit in a register?",
        "reference": "Clear a bit by ANDing the register with the inverted bit mask, such as register &= ~(1U << pin).",
        "required_concepts": [["and", "&="], ["not", "~"], ["mask", "bit"], ["clear", "zero"]]
    },
    {
        "id": 10,
        "question": "How do you toggle a specific bit?",
        "reference": "Toggle a bit using bitwise XOR with a mask, such as register ^= (1U << pin).",
        "required_concepts": [["xor", "^="], ["toggle", "invert"], ["mask", "bit"]]
    },
    {
        "id": 11,
        "question": "What is a pull-up resistor?",
        "reference": "A pull-up resistor connects an input signal to the supply voltage so the default input state is HIGH.",
        "required_concepts": [["resistor"], ["supply voltage", "vcc"], ["high"], ["default", "input"]]
    },
    {
        "id": 12,
        "question": "What is a pull-down resistor?",
        "reference": "A pull-down resistor connects an input signal to ground so the default input state is LOW.",
        "required_concepts": [["resistor"], ["ground", "gnd"], ["low"], ["default", "input"]]
    },
    {
        "id": 13,
        "question": "What is a floating input pin?",
        "reference": "A floating input pin has no defined logic level and may read unpredictable HIGH or LOW values.",
        "required_concepts": [["no defined", "undefined"], ["unpredictable", "random"], ["high"], ["low"]]
    },
    {
        "id": 14,
        "question": "What is the difference between input mode and output mode?",
        "reference": "Input mode allows a microcontroller to read an external signal, while output mode allows it to drive a signal.",
        "required_concepts": [["input", "read"], ["external signal"], ["output", "drive"], ["signal"]]
    },
    {
        "id": 15,
        "question": "What is an open-drain output?",
        "reference": "An open-drain output can pull the signal LOW but needs a pull-up resistor to produce a HIGH level.",
        "required_concepts": [["open drain"], ["low"], ["pull-up resistor"], ["high"]]
    },
    {
        "id": 16,
        "question": "What is a push-pull output?",
        "reference": "A push-pull output actively drives the pin both HIGH and LOW.",
        "required_concepts": [["push-pull"], ["high"], ["low"], ["drives", "drive"]]
    },
    {
        "id": 17,
        "question": "Why should two push-pull outputs not be connected directly together?",
        "reference": "If they drive opposite logic levels, a large current may flow and cause excessive power dissipation or damage.",
        "required_concepts": [["opposite", "different"], ["high"], ["low"], ["current"], ["damage", "short circuit"]]
    },
    {
        "id": 18,
        "question": "What does RCC stand for in STM32?",
        "reference": "RCC stands for Reset and Clock Control. It manages clock configuration and peripheral clock enabling.",
        "required_concepts": [["reset and clock control"], ["clock"], ["peripheral"]]
    },
    {
        "id": 19,
        "question": "Which register enables the GPIOA clock on STM32F401?",
        "reference": "The GPIOA clock is enabled by setting bit 0 in the RCC AHB1ENR register.",
        "required_concepts": [["ahb1enr"], ["rcc"], ["gpioa"], ["bit 0", "bit zero"]]
    },
    {
        "id": 20,
        "question": "Which register enables the GPIOB clock on STM32F401?",
        "reference": "The GPIOB clock is enabled by setting bit 1 in the RCC AHB1ENR register.",
        "required_concepts": [["ahb1enr"], ["rcc"], ["gpiob"], ["bit 1", "bit one"]]
    },
    {
        "id": 21,
        "question": "What is a memory-mapped register?",
        "reference": "A memory-mapped register is a hardware register accessed through a specific memory address.",
        "required_concepts": [["hardware register"], ["memory address"], ["accessed", "mapped"]]
    },
    {
        "id": 22,
        "question": "What is a pointer in C?",
        "reference": "A pointer is a variable that stores the memory address of another variable or object.",
        "required_concepts": [["variable"], ["memory address"], ["stores", "holds"]]
    },
    {
        "id": 23,
        "question": "Why are volatile variables used when accessing hardware registers in C?",
        "reference": "Volatile tells the compiler that a value may change unexpectedly, so it must not optimize away required reads or writes.",
        "required_concepts": [["volatile"], ["compiler"], ["change", "unexpected"], ["optimization", "optimize"], ["read", "write"]]
    },
    {
        "id": 24,
        "question": "What is the function of a bit mask?",
        "reference": "A bit mask selects specific bits in a value so they can be tested or modified.",
        "required_concepts": [["mask"], ["select", "specific"], ["bits"], ["modify", "test"]]
    },
    {
        "id": 25,
        "question": "What is a 7-segment display?",
        "reference": "A 7-segment display uses seven LED segments to display decimal digits.",
        "required_concepts": [["seven", "7"], ["segments"], ["led"], ["digits", "numbers"]]
    },
    {
        "id": 26,
        "question": "What is a common-anode 7-segment display?",
        "reference": "In a common-anode display, all LED anodes are connected together, and a segment turns on when its cathode is driven LOW.",
        "required_concepts": [["common anode"], ["anodes"], ["connected together"], ["low"], ["cathode"]]
    },
    {
        "id": 27,
        "question": "What is a common-cathode 7-segment display?",
        "reference": "In a common-cathode display, all LED cathodes are connected together, and a segment turns on when its anode is driven HIGH.",
        "required_concepts": [["common cathode"], ["cathodes"], ["connected together"], ["high"], ["anode"]]
    },
    {
        "id": 28,
        "question": "Why is a current-limiting resistor used with an LED?",
        "reference": "A current-limiting resistor restricts the current through the LED and helps prevent damage.",
        "required_concepts": [["resistor"], ["limits", "restricts"], ["current"], ["led"], ["damage", "protect"]]
    },
    {
        "id": 29,
        "question": "What is the difference between a microprocessor and a microcontroller?",
        "reference": "A microcontroller integrates a CPU, memory, and peripherals on one chip, while a microprocessor typically relies on external memory and peripherals.",
        "required_concepts": [["cpu", "processor"], ["memory"], ["peripherals"], ["one chip", "single chip"], ["external"]]
    },
    {
        "id": 30,
        "question": "What is the purpose of a timer peripheral in a microcontroller?",
        "reference": "A timer peripheral measures time intervals and can generate delays, periodic events, or PWM signals.",
        "required_concepts": [["timer"], ["time interval", "time"], ["delay"], ["periodic", "pwm"]]
    }
]

print("Total Golden Test Cases:", len(golden_cases))

Total Golden Test Cases: 30


In [9]:
def ask_model(question, max_new_tokens=150):

    prompt = f"""You are an expert assistant in Embedded Systems and STM32.

Answer the following question accurately and clearly.
If code is requested, use correct C code.

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del inputs, outputs
    gc.collect()
    torch.cuda.empty_cache()

    return answer.strip()

In [10]:
def ask_model(question, max_new_tokens=150):

    prompt = f"""You are an expert assistant in Embedded Systems and STM32.

Answer the following question accurately and clearly.
If code is requested, use correct C code.

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del inputs, outputs
    gc.collect()
    torch.cuda.empty_cache()

    return answer.strip()

In [11]:
test_answer = ask_model(
    "What is the purpose of the GPIO MODER register in STM32?"
)

print(test_answer)

The GPIO MODER (Mode) register in STM32 is used to configure the alternate functions for the General Purpose Input/Output (GPIO) pins. This register allows you to select one of the alternate functions (AF0 to AF5) for a specific GPIO pin, instead of using it as an analog input or digital output. The MODER register is part of the GPIO configuration and control block in the STM32 microcontroller.

Here's an example of how to set the alternate function for a GPIO pin in C code:

```c
#include "stm32f1xx_hal.h"

void SystemClock_Config(void);
static void MX


In [12]:
def validate_answer(answer, required_concepts):

    answer = answer.lower()

    missing_concepts = []

    for alternatives in required_concepts:

        found = any(
            term.lower() in answer
            for term in alternatives
        )

        if not found:
            missing_concepts.append(
                " / ".join(alternatives)
            )

    passed = len(missing_concepts) == 0

    return {
        "assertion_pass": passed,
        "missing_concepts": missing_concepts
    }

In [13]:
example = "GPIO MODER configures pins as input or output."

test = validate_answer(
    example,
    golden_cases[3]["required_concepts"]
)

print(test)

{'assertion_pass': False, 'missing_concepts': ['alternate function', 'analog']}


In [14]:
import json
import re

def judge_answer(question, reference_answer, model_answer):

    judge_prompt = f"""
You are an independent evaluator of technical answers.

Evaluate the model answer using the reference answer.

Question:
{question}

Reference Answer:
{reference_answer}

Model Answer:
{model_answer}

Scoring rubric:
0 = Completely incorrect or irrelevant.
1 = Mostly incorrect.
2 = Partially correct but missing important information.
3 = Mostly correct with minor missing details.
4 = Fully correct, complete, and relevant.

Rules:
- Do not reward an answer just because it is long.
- Penalize technically incorrect statements.
- Accept different wording if the meaning is correct.
- Return ONLY a valid JSON object.

Required JSON format:
{{
    "score": 0,
    "reason": "Short explanation"
}}
"""

    inputs = tokenizer(
        judge_prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    del inputs, outputs
    gc.collect()
    torch.cuda.empty_cache()

    match = re.search(
        r'\{.*?\}',
        response,
        re.DOTALL
    )

    if not match:
        return {
            "judge_score": None,
            "judge_reason": "Invalid JSON response"
        }

    try:
        data = json.loads(match.group())

        score = int(data["score"])

        if score < 0 or score > 4:
            raise ValueError("Score must be between 0 and 4")

        return {
            "judge_score": score,
            "judge_reason": data.get("reason", "")
        }

    except (ValueError, KeyError, json.JSONDecodeError):
        return {
            "judge_score": None,
            "judge_reason": "Could not parse judge response"
        }

In [15]:
test_judgement = judge_answer(
    question="What is GPIO?",
    reference_answer=golden_cases[2]["reference"],
    model_answer="GPIO allows a microcontroller to read inputs and control outputs."
)

print(test_judgement)

{'judge_score': 2, 'judge_reason': 'The answer is partially correct but missing important information.'}


In [16]:
import pandas as pd
import time

evaluation_results = []

for case in golden_cases:

    print(f"\nEvaluating Test Case {case['id']}/30")

    # Step 1: Get model answer
    answer = ask_model(case["question"])

    # Step 2: Run programmatic validator
    validation = validate_answer(
        answer,
        case["required_concepts"]
    )

    # Step 3: Run LLM judge
    judgement = judge_answer(
        case["question"],
        case["reference"],
        answer
    )

    # Step 4: Determine judge pass/fail
    judge_pass = (
        judgement["judge_score"] is not None
        and judgement["judge_score"] >= 3
    )

    # Step 5: Save results
    evaluation_results.append({

        "Test_ID": case["id"],

        "Question": case["question"],

        "Model_Answer": answer,

        "Assertion_Pass": validation["assertion_pass"],

        "Missing_Concepts": ", ".join(
            validation["missing_concepts"]
        ),

        "Judge_Score": judgement["judge_score"],

        "Judge_Reason": judgement["judge_reason"],

        "Judge_Pass": judge_pass

    })

    print("Assertion:", validation["assertion_pass"])
    print("Judge Score:", judgement["judge_score"])

    time.sleep(0.5)


results_df = pd.DataFrame(evaluation_results)

display(results_df)


Evaluating Test Case 1/30
Assertion: False
Judge Score: 4

Evaluating Test Case 2/30
Assertion: False
Judge Score: 4

Evaluating Test Case 3/30
Assertion: True
Judge Score: 3

Evaluating Test Case 4/30
Assertion: False
Judge Score: 4

Evaluating Test Case 5/30
Assertion: False
Judge Score: 4

Evaluating Test Case 6/30
Assertion: False
Judge Score: 3

Evaluating Test Case 7/30
Assertion: False
Judge Score: 4

Evaluating Test Case 8/30
Assertion: False
Judge Score: 2

Evaluating Test Case 9/30
Assertion: True
Judge Score: 4

Evaluating Test Case 10/30
Assertion: True
Judge Score: 4

Evaluating Test Case 11/30
Assertion: False
Judge Score: 2

Evaluating Test Case 12/30
Assertion: True
Judge Score: 4

Evaluating Test Case 13/30
Assertion: False
Judge Score: 1

Evaluating Test Case 14/30
Assertion: False
Judge Score: 4

Evaluating Test Case 15/30
Assertion: False
Judge Score: 4

Evaluating Test Case 16/30
Assertion: False
Judge Score: 4

Evaluating Test Case 17/30
Assertion: True
Judge Sco

,Test_ID,Question,Model_Answer,Assertion_Pass,Missing_Concepts,Judge_Score,Judge_Reason,Judge_Pass
0,1,What is an embedded system?,An Embedded System is a dedicated computer sys...,False,specific task,4,The answer provides a clear and accurate defin...,True
1,2,What is a microcontroller?,A microcontroller is a small computer on a sin...,False,single chip / chip,4,"The answer is fully correct, complete, and rel...",True
2,3,What is GPIO?,GPIO (General Purpose Input/Output) is a colle...,True,,3,The answer provides a good explanation of what...,True
3,4,What is the purpose of the GPIO MODER register...,The GPIO MODER (Mode) register in STM32 is use...,False,configures / sets,4,The answer correctly explains the purpose of t...,True
4,5,What does the GPIO ODR register do?,The GPIO Output Data Register (ODR) is a speci...,False,output state / output value,4,The answer is correct and complete.,True
5,6,What is the purpose of the GPIO IDR register?,The GPIO Input Data Register (IDR) is a read-o...,False,logic level / input state,3,"The answer is mostly correct, but it is missin...",True
6,7,What is the purpose of the GPIO BSRR register?,The GPIO Bit Set/Reset Register (BSRR) is a re...,False,atomic / independent,4,The answer is correct and complete.,True
7,8,How do you set a GPIO pin using bit manipulation?,To set a GPIO pin using bit manipulation in ST...,False,"bitwise or / |=, shift / <<",2,"The answer is partially correct, but it does n...",False
8,9,How do you clear a specific bit in a register?,"To clear a specific bit in a register, you can...",True,,4,The answer correctly explains how to clear a s...,True
9,10,How do you toggle a specific bit?,"To toggle a specific bit in an integer, you ca...",True,,4,The answer correctly explains how to toggle a ...,True


In [17]:
total_tests = len(results_df)

# Assertion results
assertion_passed = results_df["Assertion_Pass"].sum()

assertion_failed = total_tests - assertion_passed

assertion_pass_rate = (
    assertion_passed / total_tests * 100
)


# Judge results
valid_judges = results_df[
    results_df["Judge_Score"].notna()
]

judge_total = len(valid_judges)

judge_passed = (
    valid_judges["Judge_Score"] >= 3
).sum()

judge_failed = judge_total - judge_passed

judge_pass_rate = (
    judge_passed / judge_total * 100
    if judge_total > 0 else 0
)


print("========== EVALUATION REPORT ==========")

print("Total Test Cases:", total_tests)

print("\n--- Programmatic Assertions ---")

print("Passed:", assertion_passed)
print("Failed:", assertion_failed)

print(f"Pass Rate: {assertion_pass_rate:.2f}%")

print("\n--- LLM-as-a-Judge ---")

print("Valid Judgements:", judge_total)
print("Passed:", judge_passed)
print("Failed:", judge_failed)

print(f"Pass Rate: {judge_pass_rate:.2f}%")

========== EVALUATION REPORT ==========
Total Test Cases: 30

--- Programmatic Assertions ---
Passed: 10
Failed: 20
Pass Rate: 33.33%

--- LLM-as-a-Judge ---
Valid Judgements: 30
Passed: 27
Failed: 3
Pass Rate: 90.00%


In [18]:
results_df.to_csv(
    "milestone2_evaluation_results.csv",
    index=False
)

with open(
    "milestone2_evaluation_report.json",
    "w"
) as file:

    report = {
        "total_tests": total_tests,
        "assertion_passed": int(assertion_passed),
        "assertion_failed": int(assertion_failed),
        "assertion_pass_rate": assertion_pass_rate,
        "judge_valid_count": int(judge_total),
        "judge_passed": int(judge_passed),
        "judge_failed": int(judge_failed),
        "judge_pass_rate": judge_pass_rate
    }

    json.dump(report, file, indent=4)


print("Evaluation results saved successfully!")

Evaluation results saved successfully!


In [19]:
from google.colab import files

files.download("milestone2_evaluation_results.csv")

files.download("milestone2_evaluation_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# Select 5 answers for manual review

calibration_samples = results_df.head(5).copy()

calibration_samples[
    [
        "Test_ID",
        "Question",
        "Model_Answer",
        "Judge_Score",
        "Judge_Reason"
    ]
]

,Test_ID,Question,Model_Answer,Judge_Score,Judge_Reason
0,1,What is an embedded system?,An Embedded System is a dedicated computer sys...,4,The answer provides a clear and accurate defin...
1,2,What is a microcontroller?,A microcontroller is a small computer on a sin...,4,"The answer is fully correct, complete, and rel..."
2,3,What is GPIO?,GPIO (General Purpose Input/Output) is a colle...,3,The answer provides a good explanation of what...
3,4,What is the purpose of the GPIO MODER register...,The GPIO MODER (Mode) register in STM32 is use...,4,The answer correctly explains the purpose of t...
4,5,What does the GPIO ODR register do?,The GPIO Output Data Register (ODR) is a speci...,4,The answer is correct and complete.


In [21]:
# Example values only.
# Replace these scores with your own human ratings.

human_scores = [4, 3, 3, 2, 4]

calibration_samples["Human_Score"] = human_scores

calibration_samples["Score_Difference"] = abs(
    calibration_samples["Human_Score"]
    - calibration_samples["Judge_Score"]
)

display(calibration_samples[
    [
        "Test_ID",
        "Human_Score",
        "Judge_Score",
        "Score_Difference"
    ]
])

,Test_ID,Human_Score,Judge_Score,Score_Difference
0,1,4,4,0
1,2,3,4,1
2,3,3,3,0
3,4,2,4,2
4,5,4,4,0


In [22]:
mae = calibration_samples[
    "Score_Difference"
].mean()

print("Judge Calibration MAE:", round(mae, 3))

Judge Calibration MAE: 0.6


In [23]:
MIN_ASSERTION_PASS_RATE = 80
MIN_JUDGE_PASS_RATE = 80

if (
    assertion_pass_rate >= MIN_ASSERTION_PASS_RATE
    and judge_pass_rate >= MIN_JUDGE_PASS_RATE
):

    print("EVALUATION PASSED")

else:

    print("EVALUATION FAILED")

EVALUATION FAILED
